# 07 - Study 2: LLM screening (Run 5, Arm 3)

An LLM reads a job description and a biography together and judges the fit.
This notebook takes the ratings produced on a GPU, checks that they are valid,
and measures how far the counterfactual wording changes the judgement.

- **Question:** Does the manipulation move the judgement of a screening model,
  and do T1 and T2 of notebook 06 hold in this arm?
- **Expectation:** as in notebook 06 - removing communal wording raises the
  judgement and removing agentic wording lowers it; T1 expects the gap between
  women and men to grow with the communal wording of the advertisement, T2
  expects the slope of a biography over the advertisements to grow with the
  balance of that biography. Each test at alpha = 0.025.
- **Status:** third arm of Run 5. The pilot was rated on 2026-10-12 and carries
  the checks of the measurement only; the main run and its two tests were fixed
  on 2026-10-12, before any change of this arm was looked at.
- **Unit and test:** pair of advertisement and biography; the value is
  $\log P(\mathrm{Yes}) - \log P(\mathrm{No})$, the change is tested with
  Wilcoxon signed-rank, and T1 and T2 with a Spearman correlation within the
  occupation and a permutation test stratified by occupation, 5,000
  permutations, seed 42.
- **Inputs:** the ratings of the chosen run under `outputs/study2_run5/llm/v2/`
  or `outputs/study2_run5/llm/main/`, the pair file of that run,
  `outputs/study2_run5/llm/kaggle_bios.parquet` (the texts that Arm 3 was
  given), `outputs/study2_run5/arm2/` (the embeddings of Arm 2), the embedding
  cache of notebook 06 under `outputs/study2_cache/`, and from the full run of
  notebook 06 `study2_delta_r5_pool.csv`, `study2_delta_r5_pool_arm2.csv`,
  `study2_jd_level.csv`, `study2_fit_bio_slopes.csv`,
  `manipulation_check.csv`.
- **Outputs:** 19 tables under `outputs/study2_run5/arm3/<pilot or main>/`, so
  the two runs stand side by side.
- **Run:** from a fresh kernel, top to bottom, from the project root, **after
  the full run of notebook 06**. The gate, the densities of the
  advertisements, the balance of the biographies and the changes of Arm 1 and
  Arm 2 all come from notebook 06.
- **Decisions:** D1 to D7, fixed on 2026-10-11 (methodology appendix); the
  analysis runs on version 2 of the ratings only, and the run that is analysed
  is chosen by `ARM3_RUN` in the settings.

**Two rating runs**, selected by `ARM3_RUN`:

| | pilot | main |
|---|---|---|
| job descriptions per occupation | 8 | 20 |
| biographies per occupation | 25 | 100 |
| changed biographies per occupation and arm | up to 25 | up to 40 |
| pairs | 6,888 | 49,450 |
| carries | the checks of the measurement | the two tests fixed in advance |

The pilot shows that the measurement holds: the answers are repeatable, the
model separates the occupations, and the texts are the ones notebook 06
measured. It carries no inference of its own, because 8 advertisements per
occupation cannot carry a test that conditions on a property of the
advertisement. The main run widens the sample so that T1 and T2 of notebook 06
have an analogue in this arm.

**The sample**, drawn by `scripts/export_llm_inputs.py` (pilot) and
`scripts/export_llm_pairs_main.py` (main) with seed 42, per occupation:

- the advertisements of the pilot are the first 8 of the main run, so the two
  runs overlap by construction;
- the biographies the manipulation changed carry the change, in the original
  form and in the changed form, against every advertisement of the run;
- the originals of up to 50 women and 50 men carry the gap between women and
  men and the slope over the advertisements (main run only; in the pilot the
  same 25 biographies carry both);
- 10 original biographies of other occupations per advertisement carry the
  check of the validity;
- 50 pairs drawn again carry the check of the repeatability.


## 1. Settings

In [1]:
# All parameters of the analysis, one comment per parameter.
import os
os.environ.setdefault("PYTHONUTF8", "1")

from pathlib import Path as _Path
if _Path.cwd().name == "notebooks":
    os.chdir("..")

RUN_TAG      = "run5"       # output folder of this run
# Which rating run is read. The pilot of 2026-10-12 holds 8 advertisements and
# 25 biographies per occupation and carries the checks of the measurement; the
# main run holds 20 advertisements and 100 biographies per occupation and
# carries the two tests fixed in advance.
ARM3_RUN     = "main"       # "pilot" or "main"
SMOKE        = False        # True reads the test outputs of notebook 06
PRIMARY_GATE = 0.90         # content-preservation gate of the main analysis
AUC_MINIMUM  = 0.75         # below this the arm is not interpreted
N_PERM       = 5000         # permutations of the pooled test
SEED         = 42           # seed of every random step
ALPHA_PRE    = 0.025        # the two tests fixed in advance, 0.05 split over them

if ARM3_RUN not in ("pilot", "main"):
    raise AssertionError(f'ARM3_RUN is {ARM3_RUN!r}; "pilot" or "main" expected.')
# The pilot carries no inference of its own: it was rated to check the
# measurement, and its 8 advertisements per occupation cannot carry a test that
# conditions on a property of the advertisement.
INFERENCE    = ARM3_RUN == "main"
T1_T2_STATUS = "pre-specified" if INFERENCE else "descriptive, no inference"
DELTA_STATUS = ("secondary, fixed in advance" if INFERENCE
                else "descriptive, no inference")

# The embeddings of Arm 1 for the comparison of the curves. They always come
# from the full run of notebook 06, as the ratings do.
CACHE_BASE    = "outputs/study2_cache"
PRIMARY_MODEL = "sentence-transformers/all-mpnet-base-v2"
# The rule set of the manipulation, for the control of the texts.
RULES_FILE       = "config/substitution_rules_v2.csv"
FIXED_TERMS_FILE = "config/fixed_terms_v2.txt"
VALIDATION_FILE  = "config/substitution_rules_v2_validation.csv"

# Expected shape of the rating file. How many pairs and how many distinct
# prompts a run has to hold is read from the pair file of that run, so it is not
# written in here. The counts below are what the pilot produced; they are the
# controls of that run.
N_RATINGS_PER_PAIR  = 3     # passes per pair, because the repeatability rule failed
# Fixed before any result: the value of a pair is the mean of its usable passes.
PILOT_COUNTS = {
    "with_value": 6881,     # pairs that carry a value
    "partial": 8,           # of those, with fewer than three usable passes
    "without_value": 7,     # pairs with no usable pass, dropped
    "icc_n": 6873,          # pairs with all three passes, the n of the ICC
}

print(f"run {ARM3_RUN}  inference {INFERENCE}  alpha of the two tests "
      f"{ALPHA_PRE}")
print(f"gate {PRIMARY_GATE}  AUC minimum {AUC_MINIMUM}  permutations {N_PERM}  seed {SEED}")

run main  inference True  alpha of the two tests 0.025
gate 0.9  AUC minimum 0.75  permutations 5000  seed 42


## 2. Imports

In [2]:
# Libraries, the shared study module, and the input paths.
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import logsumexp
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
import statsmodels.formula.api as smf
import sys
sys.path.insert(0, str(Path.cwd()))

import importlib.util
_spec = importlib.util.spec_from_file_location("study2_core", Path("src/study2_core.py"))
study2_core = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(study2_core)

# The pilot wrote into llm/v2, the main run into llm/main. Both folders hold
# the same three file names, so only the folder changes.
LLM_ROOT = Path(f"outputs/study2_{RUN_TAG}/llm")
LLM_DIR  = LLM_ROOT / ("v2" if ARM3_RUN == "pilot" else "main")
PAIR_FILE = (LLM_ROOT / "llm_inputs_pairs.parquet" if ARM3_RUN == "pilot"
             else LLM_ROOT / "main" / "llm_inputs_pairs.parquet")
# The ratings of the other run, for the comparison between the two sessions.
OTHER_RAW_FILE = (LLM_ROOT / ("main" if ARM3_RUN == "pilot" else "v2")
                  / "llm_raw_answers.parquet")
# The ratings always come from the full sample; only the tables of notebook 06
# move to its test folder.
OUT_BASE = Path(f"outputs/study2_{RUN_TAG}/smoke13" if SMOKE
                else f"outputs/study2_{RUN_TAG}")
# The results of this arm, one folder per rating run, so the pilot and the main
# run stand side by side.
ARM3_OUT = OUT_BASE / "arm3" / ARM3_RUN
print(f"reading the ratings of the {ARM3_RUN} run from {LLM_DIR}")
print(f"writing the results to {ARM3_OUT}")
print(f"reading the tables of notebook 06 from {OUT_BASE}")
RAW_FILE    = LLM_DIR / "llm_raw_answers.parquet"
REPEAT_FILE = LLM_DIR / "llm_repeatability.parquet"
META_FILE   = LLM_DIR / "llm_run_metadata.json"
ARM1_FILE   = OUT_BASE / "study2_delta_r5_pool.csv"
ARM2_FILE   = OUT_BASE / "study2_delta_r5_pool_arm2.csv"
JD_LEVEL    = OUT_BASE / "study2_jd_level.csv"
MANIP_CHECK_FILE = OUT_BASE / "manipulation_check.csv"
BIO_SLOPES_FILE  = OUT_BASE / "study2_fit_bio_slopes.csv"
# The texts that Arm 3 was given, and the embeddings of Arm 2.
BIO_TEXT_FILE = Path(f"outputs/study2_{RUN_TAG}/llm/kaggle_bios.parquet")
ARM2_BIO_FILE = Path(f"outputs/study2_{RUN_TAG}/arm2/arm2_bio_embeddings.parquet")
ARM2_JD_FILE  = Path(f"outputs/study2_{RUN_TAG}/arm2/arm2_jd_embeddings.parquet")

for _path in (RAW_FILE, REPEAT_FILE, META_FILE, PAIR_FILE):
    if not _path.exists():
        raise FileNotFoundError(
            f"{_path} not found. Run version 2 of 07a_llm_scoring_kaggle.ipynb on "
            f"Kaggle, then scripts/resave_kaggle_parquet.py, and place the files "
            f"in {LLM_DIR}.")
for _path in (BIO_TEXT_FILE, ARM2_BIO_FILE, ARM2_JD_FILE):
    if not _path.exists():
        raise FileNotFoundError(
            f"{_path} not found. The controls of the validity and the comparison "
            f"of the arms read the texts that Arm 3 was given and the embeddings "
            f"of Arm 2; run scripts/export_llm_inputs.py and "
            f"06a_arm2_embeddings_kaggle.ipynb first.")
for _path in (ARM1_FILE, ARM2_FILE, JD_LEVEL, MANIP_CHECK_FILE,
              BIO_SLOPES_FILE):
    if not _path.exists():
        raise FileNotFoundError(
            f"{_path} not found. This notebook needs the outputs of the full run "
            f"of notebook 06; run that first.")
print("inputs found")

reading the ratings of the main run from outputs\study2_run5\llm\main
writing the results to outputs\study2_run5\arm3\main
reading the tables of notebook 06 from outputs\study2_run5
inputs found


## 3. Ratings

The file of the rating run, its shape against the pair file that was written
before it, and why each pair is in the sample.

In [3]:
# Load the ratings and check the shape of the file.
raw = pd.read_parquet(RAW_FILE)
df_repeat = pd.read_parquet(REPEAT_FILE)
meta = json.loads(META_FILE.read_text(encoding="utf-8"))

print(f"model              : {meta.get('model')}")
print(f"model digest       : {meta.get('model_digest')}")
print(f"ollama version     : {meta.get('ollama_version')}")
print(f"temperature {meta.get('temperature')}  seed {meta.get('seed')}  "
      f"num_predict {meta.get('num_predict')}  num_ctx {meta.get('num_ctx')}  "
      f"top_logprobs {meta.get('top_logprobs')}")
print(f"score              : {meta.get('score')}")
print()

# How many pairs and how many distinct prompts this run has to hold comes from
# its pair file, which was written before the run.
pairs = pd.read_parquet(PAIR_FILE)
_repeated = pairs["repeat_of"].astype(str) != ""
_checks = [
    ("main pairs", len(raw), int((~_repeated).sum())),
    ("repeated pairs", len(df_repeat), int(_repeated.sum())),
    ("distinct prompts", int(raw["prompt_hash"].nunique()),
     int(pairs.loc[~_repeated, "prompt_hash"].nunique())),
]
print(f"pair file: {PAIR_FILE.name}, {len(pairs)} rows")
for _label, _value, _expected in _checks:
    print(f"control {_label}: {_value} (expected {_expected})")
_wrong = [(a, b, c) for a, b, c in _checks if b != c]

# The value of a pair is the mean of its usable passes, so the counts below come
# from the passes themselves and not from the stored missing column, which only
# describes the last pass.
raw["n_finite_passes"] = [
    sum(1 for v in json.loads(values) if v == v) for values in raw["logit_values"]]
raw["has_value"] = raw["n_finite_passes"] >= 1
_n_with_value = int(raw["has_value"].sum())
_n_partial = int(((raw["n_finite_passes"] >= 1)
                  & (raw["n_finite_passes"] < N_RATINGS_PER_PAIR)).sum())
_n_without_value = int((raw["n_finite_passes"] == 0).sum())
_n_complete = int((raw["n_finite_passes"] == N_RATINGS_PER_PAIR).sum())
# How many pairs carry a value is known only after a run. For the pilot the
# four counts were written down before this analysis and are controls; for the
# main run they are reported, together with the identities that have to hold.
_counts = (("pairs with a value", _n_with_value, "with_value"),
           ("of those with fewer than three passes", _n_partial, "partial"),
           ("pairs without a value", _n_without_value, "without_value"),
           ("pairs with all three passes", _n_complete, "icc_n"))
for _label, _value, _key in _counts:
    if ARM3_RUN == "pilot":
        print(f"control {_label}: {_value} (expected {PILOT_COUNTS[_key]})")
        _checks.append((_label, _value, PILOT_COUNTS[_key]))
    else:
        print(f"{_label}: {_value}")
if _n_with_value + _n_without_value != len(raw):
    raise AssertionError(
        f"{_n_with_value} pairs with a value and {_n_without_value} without do "
        f"not add up to the {len(raw)} pairs of the file.")
N_ICC_EXPECTED = PILOT_COUNTS["icc_n"] if ARM3_RUN == "pilot" else _n_complete
_wrong = [(a, b, c) for a, b, c in _checks if b != c]
print(f"control value of a pair: the mean of its usable passes")
print(f"control pairs dropped from the analysis: {_n_without_value} "
      f"(those with no usable pass)")
print(f"control stored missing column, last pass only: "
      f"{int(raw['missing'].sum())} (not used for the analysis)")

# The pilot numbers its pairs with P, the main run with M.
_expected_prefix = "P" if ARM3_RUN == "pilot" else "M"
_prefixes = set(raw["pair_id"].str[0])
print(f"control main pair_id prefixes: {sorted(_prefixes)} "
      f"(expected ['{_expected_prefix}'])")
print(f"control repeated pair_id prefixes: "
      f"{sorted(set(df_repeat['pair_id'].str[0]))} (expected ['R'])")
print(f"control repeat_of empty in the main file: "
      f"{int((raw['repeat_of'].astype(str) == '').sum())} of {len(raw)} "
      f"(expected {len(raw)})")
_ratings_per_pair = sorted(set(raw["n_ratings"]))
print(f"control ratings per pair: {_ratings_per_pair} "
      f"(expected [{N_RATINGS_PER_PAIR}])")
print(f"control version: {meta.get('version')} (expected 2)")

if _wrong or _prefixes != {_expected_prefix} \
        or set(df_repeat["pair_id"].str[0]) != {"R"} \
        or _ratings_per_pair != [N_RATINGS_PER_PAIR] \
        or int(meta.get("version", 1)) < 2:
    raise AssertionError(
        f"The rating file does not have the expected shape: {_wrong}. The "
        f"analysis is fixed to version 2 of the ratings.")
# Why a pair is in the sample. The main pair file records it; for the pilot it
# follows from the sample: every own biography was changed in at least one arm.
if "role" in pairs.columns:
    raw = raw.merge(pairs[["pair_id", "role"]], on="pair_id", how="left")
    _no_role = int(raw["role"].isna().sum())
    print(f"control pairs without a role: {_no_role} (expected 0)")
    if _no_role:
        raise AssertionError(
            f"{_no_role} rated pairs do not appear in {PAIR_FILE.name}.")
else:
    raw["role"] = np.where(raw["cross_occupation"], "validity", "delta")
    print("the pair file of this run records no role; every own biography was "
          "changed in at least one arm, so it carries the change, and the "
          "biographies of other occupations carry the validity")
print(f"pairs per role: {dict(raw['role'].value_counts())}")
print()
print("the file has the expected shape")

model              : qwen3:8b
model digest       : 500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41
ollama version     : ollama version is 0.40.3
temperature 0  seed 42  num_predict 1  num_ctx 4096  top_logprobs 20
score              : logit(s) = logsumexp(yes logprobs) - logsumexp(no logprobs), not clipped

pair file: llm_inputs_pairs.parquet, 49450 rows
control main pairs: 49400 (expected 49400)
control repeated pairs: 50 (expected 50)
control distinct prompts: 48480 (expected 48480)
pairs with a value: 49323
of those with fewer than three passes: 53
pairs without a value: 77
pairs with all three passes: 49270
control value of a pair: the mean of its usable passes
control pairs dropped from the analysis: 77 (those with no usable pass)
control stored missing column, last pass only: 105 (not used for the analysis)
control main pair_id prefixes: ['M'] (expected ['M'])
control repeated pair_id prefixes: ['R'] (expected ['R'])
control repeat_of empty in the main file: 4940

## 4. Validity

- **Question:** Do the ratings carry a reliable signal about fit at all, and are
  they the ratings of the texts that notebook 06 measured?
- **Test:** job description; area under the curve for biographies of the
  advertised occupation against biographies of other occupations, plus
  repeatability, the count of unusable answers, and the hash of every biography
  text against the original and the manipulated version of notebook 06.
- **Status:** control, fixed 2026-10-11; the control of the texts added
  2026-10-12. The arm is interpreted only if the mean area under the curve
  reaches 0.75.
- **Output:** `study2_arm3_validity.csv`, `study2_arm3_auc_by_jd.csv`

In [4]:
# The value recomputed from the stored logprobs, and the unusable answers.
_PUNCTUATION = """_.,"'"""

def _logit_from_tokens(tokens_json):
    _yes, _no = [], []
    for _entry in json.loads(tokens_json):
        _token = str(_entry["token"]).strip().strip(_PUNCTUATION).lower()
        if _token == "yes":
            _yes.append(_entry["logprob"])
        elif _token == "no":
            _no.append(_entry["logprob"])
    if not _yes or not _no:
        return np.nan
    return float(logsumexp(_yes) - logsumexp(_no))

raw["logit_recomputed"] = [_logit_from_tokens(t) for t in raw["top_tokens"]]
_both = raw[["logit_s", "logit_recomputed"]].dropna()
_dev = (_both["logit_s"] - _both["logit_recomputed"]).abs()
print(f"control value against the logprobs of the last pass: largest deviation "
      f"{_dev.max():.4f}")
print(f"  the stored value is the mean of {N_RATINGS_PER_PAIR} passes, so a "
      f"deviation of the size of the within-pair spread is expected")
print()
# The counts come from the passes, which were checked when the file was loaded.
_n_outside = int(raw["answer_outside_yes_no"].sum())
print(f"pairs in the file                      : {len(raw)}")
print(f"distinct prompts among them            : {raw['prompt_hash'].nunique()}")
print(f"pairs that carry a value               : {_n_with_value}")
print(f"  of those with fewer than three passes: {_n_partial}")
print(f"pairs without a usable pass, dropped   : {_n_without_value}")
print(f"answers outside Yes/No, last pass only : {_n_outside} "
      f"({_n_outside / len(raw):.2%})")
print(f"distribution of usable passes          : "
      f"{dict(sorted(raw['n_finite_passes'].value_counts().items()))}")
print(f"  a pair with one or two usable passes still carries a value, the mean")
print(f"  over those passes; only the {_n_without_value} without any are dropped")
print(f"control truncated prompts: "
      f"{int((raw['prompt_tokens'] >= meta['num_ctx']).sum())} (expected 0)")
_finite = raw["logit_s"].dropna()
print(f"value: min {_finite.min():.3f}  median {_finite.median():.3f}  "
      f"max {_finite.max():.3f}")

_validity_rows = [
    {"check": "pairs_with_value", "n": _n_with_value,
     "value": _n_with_value / len(raw)},
    {"check": "pairs_with_partial_passes", "n": _n_partial,
     "value": _n_partial / len(raw)},
    {"check": "pairs_without_value", "n": _n_without_value,
     "value": _n_without_value / len(raw)},
    {"check": "answers_outside_yes_no", "n": _n_outside,
     "value": _n_outside / len(raw)},
]

control value against the logprobs of the last pass: largest deviation 0.7552
  the stored value is the mean of 3 passes, so a deviation of the size of the within-pair spread is expected

pairs in the file                      : 49400
distinct prompts among them            : 48480
pairs that carry a value               : 49323
  of those with fewer than three passes: 53
pairs without a usable pass, dropped   : 77
answers outside Yes/No, last pass only : 0 (0.00%)
distribution of usable passes          : {0: 77, 1: 28, 2: 25, 3: 49270}
  a pair with one or two usable passes still carries a value, the mean
  over those passes; only the 77 without any are dropped
control truncated prompts: 0 (expected 0)
value: min -40.122  median -27.677  max 27.318


In [5]:
# Repeatability: the 50 pairs rated a second time, and the spread within a pair.
_identical = int((df_repeat["abs_difference"] < 1e-6).sum())
print(f"repeated pairs          : {len(df_repeat)}")
print(f"identical to 1e-6       : {_identical}")
print(f"largest difference      : {df_repeat['abs_difference'].max():.4f}")
print(f"control repeatability rule: {_identical} of {len(df_repeat)} identical "
      f"(expected at least 49)")
if _identical < 49:
    print(f"  the rule fixed in advance was not reached, so every pair was rated "
          f"{N_RATINGS_PER_PAIR} times and the mean is used")
    _icc_n = _n_complete
    print(f"  control n of the intraclass correlation: {_icc_n} "
          f"(expected {N_ICC_EXPECTED})")
    print(f"  intraclass correlation over the passes: {meta.get('icc'):.4f}")
    print(f"    n = {_icc_n}, that is the pairs for which all "
          f"{N_RATINGS_PER_PAIR} passes returned a value; the one-way "
          f"intraclass correlation needs a complete set of passes per pair, so "
          f"the {len(raw) - _icc_n} pairs with a shorter set stay out of it")
    print(f"  mean standard deviation within a pair : {raw['logit_sd'].mean():.4f}")
    print(f"  median standard deviation within a pair: {raw['logit_sd'].median():.4f}")

# Does the change exceed the noise of the measurement? The standard deviation of
# the mean of three ratings is the spread within a pair divided by root three.
_sd_of_mean = (raw["logit_sd"] / np.sqrt(N_RATINGS_PER_PAIR)).median()
print()
print(f"median standard deviation of the mean of {N_RATINGS_PER_PAIR} ratings: "
      f"{_sd_of_mean:.4f}")
print(f"  this is a spread in the units of the value, not a ratio: the spread")
print(f"  within a pair divided by the root of {N_RATINGS_PER_PAIR}, taken as the")
print(f"  median over the pairs. Section 5 divides the median absolute change by")
print(f"  it to say how far the changes clear the noise.")
_validity_rows.append({"check": "repeatability_identical", "n": _identical,
                       "value": _identical / len(df_repeat)})
_validity_rows.append({"check": "icc", "n": len(df_repeat),
                       "value": meta.get("icc")})
_validity_rows.append({"check": "sd_of_mean_median", "n": len(raw),
                       "value": float(_sd_of_mean)})

repeated pairs          : 50
identical to 1e-6       : 0
largest difference      : 1.2083
control repeatability rule: 0 of 50 identical (expected at least 49)
  the rule fixed in advance was not reached, so every pair was rated 3 times and the mean is used
  control n of the intraclass correlation: 49270 (expected 49270)
  intraclass correlation over the passes: 0.9995
    n = 49270, that is the pairs for which all 3 passes returned a value; the one-way intraclass correlation needs a complete set of passes per pair, so the 130 pairs with a shorter set stay out of it
  mean standard deviation within a pair : 0.2087
  median standard deviation within a pair: 0.1759

median standard deviation of the mean of 3 ratings: 0.1016
  this is a spread in the units of the value, not a ratio: the spread
  within a pair divided by the root of 3, taken as the
  median over the pairs. Section 5 divides the median absolute change by
  it to say how far the changes clear the noise.


In [6]:
# The pilot and the main run against each other, on the prompts that both rated.
# The model answers with temperature 0 and a fixed seed, so two sessions should
# agree; the intraclass correlation between the sessions says how far they do.
_other_run = "pilot" if ARM3_RUN == "main" else "main"
_other_raw = OTHER_RAW_FILE
if not _other_raw.exists():
    print(f"the ratings of the {_other_run} run are not in {_other_raw}, so the "
          f"two sessions cannot be compared; this cell needs both runs")
    df_arm3_sessions = pd.DataFrame(
        [{"other_run": _other_run, "n_prompts": 0, "icc_between_sessions": np.nan,
          "note": "the other run is not present"}])
else:
    _other = pd.read_parquet(_other_raw)[["prompt_hash", "logit_s"]]
    _other = _other.dropna().groupby("prompt_hash")["logit_s"].mean()
    _here = raw[["prompt_hash", "logit_s"]].dropna().groupby(
        "prompt_hash")["logit_s"].mean()
    _shared = sorted(set(_here.index) & set(_other.index))
    print(f"prompts rated in both runs: {len(_shared)}")
    if len(_shared) < 10:
        raise AssertionError(
            f"Only {len(_shared)} prompts were rated in both runs; the "
            f"comparison between the sessions needs at least 10.")
    _a = _here.reindex(_shared).to_numpy(float)
    _b = _other.reindex(_shared).to_numpy(float)
    _difference = np.abs(_a - _b)
    # One-way intraclass correlation over the two sessions of every prompt.
    _matrix = np.c_[_a, _b]
    _row_means = _matrix.mean(axis=1)
    _grand = float(_matrix.mean())
    _k = _matrix.shape[1]
    _ms_between = float(((_row_means - _grand) ** 2).sum() * _k
                        / (len(_shared) - 1))
    _ms_within = float(((_matrix - _row_means[:, None]) ** 2).sum()
                       / (len(_shared) * (_k - 1)))
    _icc = ((_ms_between - _ms_within) / (_ms_between + (_k - 1) * _ms_within)
            if (_ms_between + (_k - 1) * _ms_within) > 0 else float("nan"))
    _rho, _p = spearmanr(_a, _b)
    print(f"identical to 1e-6        : {int((_difference < 1e-6).sum())} of "
          f"{len(_shared)}")
    print(f"largest difference       : {_difference.max():.4f}")
    print(f"median difference        : {float(np.median(_difference)):.4f}")
    print(f"intraclass correlation   : {_icc:.4f}")
    print(f"rank correlation         : {float(_rho):+.4f}  p {float(_p):.4g}")
    df_arm3_sessions = pd.DataFrame([{
        "other_run": _other_run, "n_prompts": len(_shared),
        "n_identical_1e6": int((_difference < 1e-6).sum()),
        "max_difference": float(_difference.max()),
        "median_difference": float(np.median(_difference)),
        "icc_between_sessions": float(_icc),
        "spearman_rho": float(_rho), "spearman_p": float(_p),
        "note": "mean over the passes of a prompt in each run"}])

prompts rated in both runs: 5775
identical to 1e-6        : 3777 of 5775
largest difference       : 5.1767
median difference        : 0.0000
intraclass correlation   : 0.9992
rank correlation         : +0.9990  p 0


In [7]:
# Control: every biography text that Arm 3 was given is the text of notebook 06.
# The original versions are compared with the pool biographies of
# df_features.parquet, matched by the row number that the pool carries as
# bio_id. The manipulated versions are produced again here with the v2 list and
# the rule set after D7, and compared by the hash of the text.
import hashlib
import spacy
import src.substitution_v2 as _sub_v2
from src.lexicon import load_lexicons as _load_lex

def _sha(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()

_bio_texts = pd.read_parquet(BIO_TEXT_FILE)
_rated_keys = set(raw["bio_key"])
_bio_texts = _bio_texts[_bio_texts["bio_key"].isin(_rated_keys)]
print(f"biography versions that Arm 3 rated: {len(_bio_texts)} "
      f"({dict(_bio_texts['version'].value_counts())})")

_features = pd.read_parquet("outputs/df_features.parquet",
                            columns=["text_clean", "occupation", "gender_label"])
print(f"control row numbers of df_features.parquet unique: "
      f"{_features.index.is_unique} (expected True)")
if not _features.index.is_unique:
    raise AssertionError(
        "The row numbers of df_features.parquet are not unique, so a biography "
        "cannot be identified by bio_id.")
_originals = _bio_texts[_bio_texts["version"] == "original"]
_missing_ids = [int(b) for b in _originals["bio_id"] if b not in _features.index]
print(f"control biographies of the file that df_features.parquet does not hold: "
      f"{len(_missing_ids)} (expected 0)")
if _missing_ids:
    raise AssertionError(
        f"{len(_missing_ids)} biographies carry a row number that "
        f"df_features.parquet does not hold, for example {_missing_ids[:5]}.")
_orig_wrong = [row.bio_key for row in _originals.itertuples()
               if _sha(row.bio_text) != _sha(_features.loc[row.bio_id, "text_clean"])]
print(f"control original texts with the hash of the pool biography: "
      f"{len(_originals) - len(_orig_wrong)} of {len(_originals)} (expected all)")
if _orig_wrong:
    raise AssertionError(
        f"{len(_orig_wrong)} original texts differ from the pool biography of "
        f"notebook 06, for example {_orig_wrong[:5]}.")

# The manipulated versions are produced again, with the same rules and the same
# parse that notebook 06 uses.
_nlp_v2 = spacy.load("en_core_web_lg")
_lex_a = _load_lex(version="A")
_lex_ab = _load_lex(version="AB")
_rules_v2 = _sub_v2.load_rules(RULES_FILE)
_fixed_terms_v2 = _sub_v2.load_fixed_terms(FIXED_TERMS_FILE)
# check_rules picks the replacement of every rule, the first candidate that is
# neutral under A u B, and sets it in place. Without this step no rule has a
# replacement and the engine would produce the original text again, so the
# chain is the one of notebook 06 and of scripts/export_llm_inputs.py.
_check_v2 = _sub_v2.check_rules(_rules_v2, _lex_a, _lex_ab)
print(f"rules whose source word is not in dictionary A (expected 0): "
      f"{len(_check_v2['source_failures'])}")
print(f"rules without a neutral replacement (occurrences stay untreated): "
      f"{len(_check_v2['no_candidate'])}")
# D7: the rules that failed the rater validation are switched off, exactly as
# notebook 06 switches them off before it manipulates.
if not Path(VALIDATION_FILE).exists():
    raise FileNotFoundError(
        f"{VALIDATION_FILE} not found. The manipulation is fixed as v2 after "
        f"the validation of D7; the texts cannot be produced again without it.")
_validation_v2 = pd.read_csv(VALIDATION_FILE)
_failed_rules = set(
    _validation_v2.loc[~_validation_v2["passed"].astype(bool), "rule_id"])
_sub_v2.apply_validation(_rules_v2, _failed_rules)
print(f"rules {len(_rules_v2)}, switched off by D7: {len(_failed_rules)} "
      f"{sorted(_failed_rules)}")
_engine_v2 = _sub_v2.SubstitutionEngine(_rules_v2, _fixed_terms_v2, _lex_a)
_manipulated = _bio_texts[_bio_texts["version"] != "original"]
print(f"manipulated versions to produce again: {len(_manipulated)}")
_wrong_manip, _checked = [], 0
for occ in sorted(_manipulated["occ"].unique()):
    _rows = _manipulated[_manipulated["occ"] == occ]
    _texts = [_features.loc[int(b), "text_clean"] for b in _rows["bio_id"]]
    _docs = list(_nlp_v2.pipe(_texts, batch_size=32))
    for _row, _doc in zip(_rows.itertuples(), _docs):
        _again = _engine_v2.apply(_doc, _row.version).manipulated
        _checked += 1
        if _sha(_again) != _sha(_row.bio_text):
            _wrong_manip.append(_row.bio_key)
print(f"control manipulated texts with the hash of the version of notebook 06: "
      f"{_checked - len(_wrong_manip)} of {_checked} (expected all)")
if _wrong_manip:
    raise AssertionError(
        f"{len(_wrong_manip)} manipulated texts differ from what the v2 list "
        f"produces today, for example {_wrong_manip[:5]}. Arm 3 then rated other "
        f"texts than notebook 06 measured.")
_validity_rows.append({"check": "bio_texts_hash_equal", "n": _checked
                       + len(_originals), "value": 1.0})
print("every biography text of Arm 3 is the text of notebook 06")

biography versions that Arm 3 rated: 3404 ({'original': np.int64(2364), 'communion': np.int64(520), 'agency': np.int64(520)})
control row numbers of df_features.parquet unique: True (expected True)
control biographies of the file that df_features.parquet does not hold: 0 (expected 0)
control original texts with the hash of the pool biography: 2364 of 2364 (expected all)
rules whose source word is not in dictionary A (expected 0): 0
rules without a neutral replacement (occurrences stay untreated): 0
rules 252, switched off by D7: 3 ['A004', 'A007', 'C005']
manipulated versions to produce again: 1040
control manipulated texts with the hash of the version of notebook 06: 1040 of 1040 (expected all)
every biography text of Arm 3 is the text of notebook 06


In [8]:
# Area under the curve per job description: own occupation against other
# occupations, on the original biographies only.
_auc_rows = []
_originals = raw[(raw["version"] == "original") & raw["has_value"]]
for jd_id, group in _originals.groupby("jd_id"):
    _own = group[~group["cross_occupation"]]
    _other = group[group["cross_occupation"]]
    if len(_own) < 5 or len(_other) < 5:
        continue
    _score = np.r_[_own["logit_s"].to_numpy(), _other["logit_s"].to_numpy()]
    if len(np.unique(_score)) < 2:
        continue
    _auc_rows.append({"jd_id": jd_id, "occ": group["occ"].iloc[0],
                      "n_own": len(_own), "n_other": len(_other),
                      "auc": float(roc_auc_score(
                          np.r_[np.ones(len(_own)), np.zeros(len(_other))], _score))})
df_arm3_auc = pd.DataFrame(_auc_rows)
if df_arm3_auc.empty:
    raise AssertionError(
        "No job description has enough biographies of both kinds for the check.")

_mean_auc = float(df_arm3_auc["auc"].mean())
print(f"job descriptions in the check: {len(df_arm3_auc)}")
print(f"mean area under the curve    : {_mean_auc:.3f}")
print(f"median                       : {df_arm3_auc['auc'].median():.3f}")
print(f"quartiles                    : {df_arm3_auc['auc'].quantile(0.25):.3f} / "
      f"{df_arm3_auc['auc'].quantile(0.75):.3f}")
print(f"range                        : {df_arm3_auc['auc'].min():.3f} to "
      f"{df_arm3_auc['auc'].max():.3f}")
print(f"share below 0.5              : {(df_arm3_auc['auc'] < 0.5).mean():.1%}")
print()
print("per occupation:")
print(df_arm3_auc.groupby("occ")["auc"].agg(["count", "mean", "min", "max"])
      .round(3).to_string())

ARM3_INTERPRETABLE = _mean_auc >= AUC_MINIMUM
print()
print(f"control mean area under the curve: {_mean_auc:.3f} "
      f"(expected at least {AUC_MINIMUM})")
if not ARM3_INTERPRETABLE:
    print("The minimum fixed in advance is not reached, so the arm is not "
          "interpreted substantively. The results below are still computed and "
          "saved, and every table carries interpretable=False.")
_validity_rows.append({"check": "mean_auc", "n": len(df_arm3_auc),
                       "value": _mean_auc})
df_arm3_validity = pd.DataFrame(_validity_rows)
df_arm3_validity["auc_minimum"] = AUC_MINIMUM
df_arm3_validity["interpretable"] = ARM3_INTERPRETABLE

job descriptions in the check: 260
mean area under the curve    : 0.838
median                       : 0.883
quartiles                    : 0.769 / 0.943
range                        : 0.356 to 0.994
share below 0.5              : 4.6%

per occupation:
                   count   mean    min    max
occ                                          
accountant            20  0.887  0.769  0.948
architect             20  0.672  0.437  0.930
attorney              20  0.895  0.739  0.974
dentist               20  0.903  0.590  0.994
dietitian             20  0.956  0.901  0.985
nurse                 20  0.905  0.801  0.979
paralegal             20  0.944  0.862  0.992
physician             20  0.915  0.738  0.990
professor             20  0.704  0.453  0.911
psychologist          20  0.697  0.356  0.958
software_engineer     20  0.875  0.677  0.924
surgeon               20  0.775  0.460  0.975
teacher               20  0.763  0.493  0.947

control mean area under the curve: 0.838 (expected at le

## 5. Change in the judgement

- **Question:** Does the counterfactual wording move the judgement of the model?
- **Test:** biography; two measures of the change, both averaged over the job
  descriptions of the occupation: the change in
  $\log P(\mathrm{Yes}) - \log P(\mathrm{No})$, and the change in the
  percentile rank of the biography among the other biographies of the same job
  description, formed as in Arm 1 and Arm 2. Signed rank test per arm, with the
  content-preservation gate as the main analysis and all biographies as a
  sensitivity.
- **Status:** main analysis, fixed 2026-10-11; the percentile rank, the switches
  of the judgement, the split at the median and the model of the length added
  2026-10-12, before any result. The split at the median is exploratory.
- **Output:** `study2_arm3_delta.csv`, `study2_arm3_delta_by_jd.csv`,
  `study2_arm3_pct.csv`, `study2_arm3_pct_by_jd.csv`, `study2_arm3_tests.csv`,
  `study2_arm3_switch.csv`, `study2_arm3_split.csv`, `study2_arm3_words.csv`

In [9]:
# The change per biography and job description, then averaged per biography.
_own = raw[(~raw["cross_occupation"]) & raw["has_value"]].copy()
_orig = (_own[_own["version"] == "original"]
         .set_index(["jd_id", "occ", "bio_idx"])["logit_s"])
_rows = []
for version in ("communion", "agency"):
    for row in _own[_own["version"] == version].itertuples():
        _key = (row.jd_id, row.occ, row.bio_idx)
        if _key not in _orig.index:
            continue
        _before = float(_orig.loc[_key])
        if not np.isfinite(_before) or not np.isfinite(row.logit_s):
            continue
        _rows.append({"occ": row.occ, "bio_idx": row.bio_idx, "arm": version,
                      "jd_id": row.jd_id, "female": row.female,
                      "logit_before": _before, "logit_after": float(row.logit_s),
                      "delta": float(row.logit_s) - _before})
df_arm3_pairs = pd.DataFrame(_rows)
print(f"biography and job description pairs with a change: {len(df_arm3_pairs)}")
print(f"  per arm: {dict(df_arm3_pairs['arm'].value_counts())}")

df_arm3_delta = (df_arm3_pairs.groupby(["occ", "bio_idx", "arm", "female"])
                 .agg(delta=("delta", "mean"), n_jds=("delta", "size"),
                      logit_before=("logit_before", "mean"))
                 .reset_index())
print(f"biographies with a change: {len(df_arm3_delta)}")
print(f"  job descriptions per biography: median {df_arm3_delta['n_jds'].median():.0f}")

# The gate of Arm 1 decides which biographies carry the main analysis.
_arm1 = pd.read_csv(ARM1_FILE)
df_arm3_delta = df_arm3_delta.merge(
    _arm1[["occ", "bio_idx", "arm", "sim_cp", "delta"]]
    .rename(columns={"delta": "delta_arm1"}),
    on=["occ", "bio_idx", "arm"], how="left")
_with_gate = int(df_arm3_delta["sim_cp"].notna().sum())
print(f"control biographies with a gate value: {_with_gate} of "
      f"{len(df_arm3_delta)}")
if SMOKE:
    print(f"  the test tables of notebook 06 hold fewer biographies, so the gate "
          f"covers only part of the pairs")
elif _with_gate != len(df_arm3_delta):
    raise AssertionError(
        f"{len(df_arm3_delta) - _with_gate} biographies have no gate value; the "
        f"tables of notebook 06 do not cover this pool.")

biography and job description pairs with a change: 20772
  per arm: {'agency': np.int64(10393), 'communion': np.int64(10379)}
biographies with a change: 1040
  job descriptions per biography: median 20
control biographies with a gate value: 1040 of 1040


In [10]:
# The value per biography and job description in one table, and the second
# measure of the change: the percentile rank, built as in Arm 1 and Arm 2. The
# manipulated biography is ranked against the original versions of the other
# biographies of the same job description, and the ranks are averaged over the
# job descriptions of the occupation.
#
# A biography that an arm did not change was not rated a second time: its prompt
# is the one of the original, and the pipeline answers an identical prompt from
# its cache. Its manipulated value is therefore its original value, and its rank
# does not move. Such biographies stay in the comparison set and carry no row of
# their own, exactly as they carry no row in the table of the change in logit.
_own_all = raw[(~raw["cross_occupation"]) & raw["has_value"]]
_pct_rows, _pct_jd_rows, _orig_rows = [], [], []
_no_full_original, _no_full_arm = [], []
for occ in sorted(_own_all["occ"].unique()):
    _occ_rows = _own_all[_own_all["occ"] == occ]
    _jds = sorted(_occ_rows["jd_id"].unique())
    _orig = (_occ_rows[_occ_rows["version"] == "original"]
             .pivot_table(index="bio_idx", columns="jd_id", values="logit_s")
             .reindex(columns=_jds))
    _n_cells = int(_orig.notna().to_numpy().sum())
    _n_rows = int((_occ_rows["version"] == "original").sum())
    if _n_cells != _n_rows:
        raise AssertionError(
            f"{occ}: the table of the original values holds {_n_cells} values "
            f"for {_n_rows} rated pairs, so a biography and job description "
            f"appears more than once.")
    _complete = _orig.dropna()
    _no_full_original.extend((occ, int(b))
                             for b in _orig.index.difference(_complete.index))
    if len(_complete) < 3:
        raise AssertionError(
            f"{occ} has {len(_complete)} biographies with a value for every job "
            f"description; at least 3 are needed for a rank.")
    _bios = list(_complete.index)
    _s_orig = _complete.to_numpy(float)
    _pct_before_jd = study2_core.pool_pct_per_jd(_s_orig)
    _female = (_occ_rows.drop_duplicates("bio_idx")
               .set_index("bio_idx")["female"].reindex(_bios))
    # The value and the rank of every original, per job description. The two
    # pre-registered tests read this table.
    for i, _bio_idx in enumerate(_bios):
        for j, _jd_id in enumerate(_jds):
            _orig_rows.append({
                "occ": occ, "bio_idx": int(_bio_idx), "jd_id": _jd_id,
                "female": int(_female.iloc[i]),
                "logit": float(_s_orig[i, j]),
                "pct_orig": float(_pct_before_jd[i, j])})
    for arm in ("communion", "agency"):
        _manip = (_occ_rows[_occ_rows["version"] == arm]
                  .pivot_table(index="bio_idx", columns="jd_id", values="logit_s")
                  .reindex(index=_bios, columns=_jds))
        _rated_any = _manip.notna().any(axis=1).to_numpy()
        _rated_all = _manip.notna().all(axis=1).to_numpy()
        _no_full_arm.extend((occ, arm, int(_bios[i])) for i in range(len(_bios))
                            if _rated_any[i] and not _rated_all[i])
        _s_manip = np.where(_manip.notna().to_numpy(),
                            _manip.to_numpy(dtype=float), _s_orig)
        _pct_after_jd = study2_core.indiv_cf_pct_per_jd(_s_orig, _s_manip)
        for i, _bio_idx in enumerate(_bios):
            if not _rated_all[i]:
                continue
            _pct_rows.append({
                "occ": occ, "bio_idx": int(_bio_idx), "arm": arm,
                "female": int(_female.iloc[i]),
                "pct_before": float(_pct_before_jd[i].mean()),
                "pct_after": float(_pct_after_jd[i].mean()),
                "delta_pct": float(_pct_after_jd[i].mean()
                                   - _pct_before_jd[i].mean()),
                "n_jds": len(_jds), "n_pool": len(_bios)})
            for j, _jd_id in enumerate(_jds):
                _pct_jd_rows.append({
                    "occ": occ, "bio_idx": int(_bio_idx), "arm": arm,
                    "jd_id": _jd_id,
                    "pct_before": float(_pct_before_jd[i, j]),
                    "pct_after": float(_pct_after_jd[i, j]),
                    "delta_pct": float(_pct_after_jd[i, j] - _pct_before_jd[i, j])})
df_arm3_pct = pd.DataFrame(_pct_rows)
df_arm3_pct_by_jd = pd.DataFrame(_pct_jd_rows)
df_arm3_originals = pd.DataFrame(_orig_rows)
print(f"biographies in the comparison set per occupation: "
      f"{sorted(set(df_arm3_pct['n_pool']))}")
print(f"job descriptions per occupation: {sorted(set(df_arm3_pct['n_jds']))}")
print(f"biographies without a value for every job description, left out of the "
      f"comparison set: {len(_no_full_original)} {_no_full_original}")
print(f"biographies rated in an arm but not for every job description, left out "
      f"of that arm: {len(_no_full_arm)} {_no_full_arm}")
print(f"rows: {len(df_arm3_pct)} biographies, {len(df_arm3_pct_by_jd)} "
      f"biography and job description pairs of the change, "
      f"{len(df_arm3_originals)} of the originals")
print(f"  per arm: {dict(df_arm3_pct['arm'].value_counts())}")
print(f"  median change in the percentile rank: "
      f"{df_arm3_pct['delta_pct'].median():+.3f} points")

# The two measures travel together, so the tests below run on both.
df_arm3_delta = df_arm3_delta.merge(
    df_arm3_pct[["occ", "bio_idx", "arm", "delta_pct", "pct_before",
                 "pct_after"]],
    on=["occ", "bio_idx", "arm"], how="left")
# A rank needs a value for every advertisement of the occupation, before and
# after; a change in logit needs one advertisement. The two measures therefore
# rest on slightly different sets of biographies, and every biography that
# carries the one but not the other has to appear in one of the two lists above.
_without_rank = df_arm3_delta[df_arm3_delta["delta_pct"].isna()]
_explained = ({(occ, bio) for occ, bio in _no_full_original}
              | {(occ, bio) for occ, arm, bio in _no_full_arm})
_unexplained = [(row.occ, int(row.bio_idx), row.arm)
                for row in _without_rank.itertuples()
                if (row.occ, int(row.bio_idx)) not in _explained]
print(f"biographies with a change in logit but none in rank: "
      f"{len(_without_rank)} of {len(df_arm3_delta)}")
if len(_without_rank):
    print(f"  they lost a value for at least one advertisement: "
          f"{sorted((r.occ, int(r.bio_idx), r.arm) for r in _without_rank.itertuples())}")
print(f"control of those that the two lists do not explain: "
      f"{len(_unexplained)} (expected 0)")
if _unexplained:
    raise AssertionError(
        f"{len(_unexplained)} biographies carry a change in logit but no "
        f"change in rank without having lost a value: {_unexplained[:5]}. The "
        f"two tables do not cover the same biographies.")

biographies in the comparison set per occupation: [94, 95, 99, 100]
job descriptions per occupation: [20]
biographies without a value for every job description, left out of the comparison set: 17 [('accountant', 171), ('architect', 13), ('architect', 98), ('architect', 132), ('architect', 196), ('architect', 203), ('architect', 235), ('professor', 15), ('professor', 95), ('professor', 106), ('professor', 117), ('professor', 167), ('software_engineer', 34), ('software_engineer', 58), ('software_engineer', 68), ('software_engineer', 203), ('software_engineer', 205)]
biographies rated in an arm but not for every job description, left out of that arm: 3 [('architect', 'communion', 121), ('architect', 'agency', 245), ('software_engineer', 'agency', 98)]
rows: 1027 biographies, 20540 biography and job description pairs of the change, 25660 of the originals
  per arm: {'agency': np.int64(516), 'communion': np.int64(511)}
  median change in the percentile rank: +0.101 points
biographies with a

In [11]:
# Per arm and per occupation, with the gate as the main analysis and all
# biographies as a sensitivity. Both measures of the change are tested: the
# change in logit and the change in the percentile rank.
_measures = (("arm3_delta", "delta", "change in logit"),
             ("arm3_delta_pct", "delta_pct", "change in percentile rank"))
_test_rows = []
_sd_of_mean = (raw["logit_sd"] / np.sqrt(N_RATINGS_PER_PAIR)).median()
for _test_name, _column, _measure_label in _measures:
    _n_measure = int(df_arm3_delta[_column].notna().sum())
    print(f"{_measure_label}: {_n_measure} of {len(df_arm3_delta)} biographies "
          f"carry it")
    print(f"  {'arm':<10} {'subset':<7} {'n':>5} {'median':>9} {'up':>8} "
          f"{'down':>8} {'p':>10}")
    print("  " + "-" * 62)
    for arm in ("communion", "agency"):
        # A biography enters a test if it carries the measure that test reads.
        _arm_frame = df_arm3_delta[(df_arm3_delta["arm"] == arm)
                                   & df_arm3_delta[_column].notna()]
        for label, frame in (
                ("gate", _arm_frame[_arm_frame["sim_cp"] >= PRIMARY_GATE]),
                ("all", _arm_frame)):
            if frame.empty:
                print(f"  {arm:<10} {label:<7} no biography in this subset, "
                      f"left out")
                continue
            _p = study2_core._safe_wilcoxon(frame[_column])
            if not np.isfinite(_p):
                print(f"  {arm:<10} {label:<7} fewer than 10 biographies with a "
                      f"change, so the signed rank test is not defined")
            _row = {"test": _test_name, "measure": _column, "arm": arm,
                    "subset": label, "level": "pooled", "n": len(frame),
                    "median_delta": float(frame[_column].median()),
                    "median_abs_delta": float(frame[_column].abs().median()),
                    "mean_delta": float(frame[_column].mean()),
                    "share_up": float((frame[_column] > 0).mean()),
                    "share_down": float((frame[_column] < 0).mean()),
                    "p_wilcoxon": float(_p)}
            if _column == "delta":
                _row["sd_of_mean"] = float(_sd_of_mean)
                _row["abs_delta_over_noise"] = (float(frame[_column].abs().median())
                                                / float(_sd_of_mean))
            print(f"  {arm:<10} {label:<7} {len(frame):>5} "
                  f"{frame[_column].median():>+9.3f} "
                  f"{(frame[_column] > 0).mean():>8.1%} "
                  f"{(frame[_column] < 0).mean():>8.1%} {_p:>10.4g}")
            _test_rows.append(_row)
    print()
    print(f"  per occupation, gate subset:")
    print(f"    {'occ':<20} {'arm':<10} {'n':>5} {'median':>9} {'up':>8} "
          f"{'p':>10}")
    _too_small = []
    for occ in sorted(df_arm3_delta["occ"].unique()):
        for arm in ("communion", "agency"):
            _s = df_arm3_delta[(df_arm3_delta["occ"] == occ)
                               & (df_arm3_delta["arm"] == arm)
                               & (df_arm3_delta["sim_cp"] >= PRIMARY_GATE)
                               & df_arm3_delta[_column].notna()]
            if len(_s) < 5:
                _too_small.append(f"{occ}/{arm}")
                continue
            _p = study2_core._safe_wilcoxon(_s[_column])
            print(f"    {occ:<20} {arm:<10} {len(_s):>5} "
                  f"{_s[_column].median():>+9.3f} "
                  f"{(_s[_column] > 0).mean():>8.1%} {_p:>10.4g}")
            _row = {"test": _test_name, "measure": _column, "arm": arm,
                    "subset": "gate", "level": occ, "n": len(_s),
                    "median_delta": float(_s[_column].median()),
                    "median_abs_delta": float(_s[_column].abs().median()),
                    "mean_delta": float(_s[_column].mean()),
                    "share_up": float((_s[_column] > 0).mean()),
                    "share_down": float((_s[_column] < 0).mean()),
                    "p_wilcoxon": float(_p)}
            if _column == "delta":
                _row["sd_of_mean"] = float(_sd_of_mean)
                _row["abs_delta_over_noise"] = (float(_s[_column].abs().median())
                                                / float(_sd_of_mean))
            _test_rows.append(_row)
    if _too_small:
        print(f"    occupation and arm with fewer than 5 biographies in the "
              f"gate subset, left out: {_too_small}")
    print()
df_arm3_tests = pd.DataFrame(_test_rows)
df_arm3_tests["interpretable"] = ARM3_INTERPRETABLE
df_arm3_tests["status"] = DELTA_STATUS
print(f"The column abs_delta_over_noise is the median absolute change in logit "
      f"divided by the standard deviation of the mean of {N_RATINGS_PER_PAIR} "
      f"ratings ({_sd_of_mean:.4f}), so it says how far the changes in logit sit "
      f"above the measurement noise. It is formed for the change in logit only; "
      f"the percentile rank is not measured in the same unit.")

change in logit: 1040 of 1040 biographies carry it
  arm        subset      n    median       up     down          p
  --------------------------------------------------------------
  communion  gate      519    -0.040    45.9%    54.1%     0.0066
  communion  all       520    -0.039    46.0%    54.0%   0.007292
  agency     gate      517    +0.022    53.8%    46.2%   0.001944
  agency     all       520    +0.022    53.7%    46.3%   0.002488

  per occupation, gate subset:
    occ                  arm            n    median       up          p
    accountant           communion     40    +0.103    55.0%     0.3682
    accountant           agency        40    +0.134    57.5%     0.1088
    architect            communion     40    +0.007    52.5%     0.3336
    architect            agency        40    -0.037    42.5%        0.1
    attorney             communion     40    +0.009    55.0%     0.7852
    attorney             agency        40    +0.109    60.0%    0.05481
    dentist       

In [12]:
# Descriptive: how often the judgement itself changes side. The value is
# log P(Yes) - log P(No), so P(Yes) is above one half exactly when the value is
# above zero.
_switch_rows = []
print(f"  {'arm':<10} {'pairs':>6} {'yes before':>11} {'yes after':>10} "
      f"{'yes to no':>10} {'no to yes':>10} {'unchanged':>10}")
print("  " + "-" * 72)
for arm in ("communion", "agency"):
    _s = df_arm3_pairs[df_arm3_pairs["arm"] == arm]
    _before = _s["logit_before"] > 0
    _after = _s["logit_after"] > 0
    _yes_to_no = int((_before & ~_after).sum())
    _no_to_yes = int((~_before & _after).sum())
    print(f"  {arm:<10} {len(_s):>6} {_before.mean():>10.1%} "
          f"{_after.mean():>9.1%} {_yes_to_no:>10} {_no_to_yes:>10} "
          f"{len(_s) - _yes_to_no - _no_to_yes:>10}")
    _switch_rows.append({"arm": arm, "n_pairs": len(_s),
                         "share_yes_before": float(_before.mean()),
                         "share_yes_after": float(_after.mean()),
                         "n_yes_to_no": _yes_to_no, "n_no_to_yes": _no_to_yes,
                         "n_unchanged_side": int(len(_s) - _yes_to_no
                                                 - _no_to_yes)})
df_arm3_switch = pd.DataFrame(_switch_rows)
df_arm3_switch["interpretable"] = ARM3_INTERPRETABLE
print()
print("The counts are pairs of one biography and one job description, not "
      "biographies: a biography is rated against every job description of its "
      "occupation.")

  arm         pairs  yes before  yes after  yes to no  no to yes  unchanged
  ------------------------------------------------------------------------
  communion   10379       7.0%      6.8%         73         50      10256
  agency      10393       7.7%      8.2%         35         87      10271

The counts are pairs of one biography and one job description, not biographies: a biography is rated against every job description of its occupation.


In [13]:
# Exploratory: the change in logit for pairs above and below the median of the
# original value. The split is made within the arm, so each half holds half of
# the pairs of that arm. This was not among the tests fixed in advance.
from scipy.stats import mannwhitneyu as _mwu_split

_split_rows = []
print(f"  {'arm':<10} {'half':<7} {'n':>6} {'median value':>13} "
      f"{'median change':>14} {'p within':>10}")
print("  " + "-" * 68)
for arm in ("communion", "agency"):
    _s = df_arm3_pairs[df_arm3_pairs["arm"] == arm]
    _median_before = float(_s["logit_before"].median())
    _halves = {"above": _s[_s["logit_before"] > _median_before],
               "below": _s[_s["logit_before"] <= _median_before]}
    for _label, _half in _halves.items():
        _p_within = study2_core._safe_wilcoxon(_half["delta"])
        if not np.isfinite(_p_within):
            print(f"  {arm:<10} {_label:<7} {len(_half):>6}: fewer than 10 pairs "
                  f"with a change, so the signed rank test is not defined")
        print(f"  {arm:<10} {_label:<7} {len(_half):>6} "
              f"{_half['logit_before'].median():>+13.3f} "
              f"{_half['delta'].median():>+14.3f} {_p_within:>10.4g}")
        _split_rows.append({"arm": arm, "half": _label, "n": len(_half),
                            "median_logit_before":
                                float(_half["logit_before"].median()),
                            "median_delta": float(_half["delta"].median()),
                            "p_wilcoxon_within": float(_p_within)})
    _u, _p_between = _mwu_split(_halves["above"]["delta"],
                                _halves["below"]["delta"],
                                alternative="two-sided")
    print(f"  {arm:<10} difference between the halves: "
          f"Mann-Whitney p={float(_p_between):.4g}")
    _split_rows.append({"arm": arm, "half": "difference",
                        "n": len(_s),
                        "median_logit_before": _median_before,
                        "median_delta": float(_halves["above"]["delta"].median()
                                              - _halves["below"]["delta"].median()),
                        "p_mannwhitney_between": float(_p_between)})
df_arm3_split = pd.DataFrame(_split_rows)
df_arm3_split["status"] = "exploratory"
df_arm3_split["interpretable"] = ARM3_INTERPRETABLE

  arm        half         n  median value  median change   p within
  --------------------------------------------------------------------
  communion  above     5187       -19.567         -0.155  3.991e-20
  communion  below     5192       -31.586         +0.004     0.5227
  communion  difference between the halves: Mann-Whitney p=6.021e-18
  agency     above     5196       -18.545         +0.116  2.661e-19
  agency     below     5197       -31.531         +0.068  2.869e-20
  agency     difference between the halves: Mann-Whitney p=0.01942


In [14]:
# Does the change in the judgement follow the change in length? The number of
# words removed or added comes from the manipulation check of notebook 06.
_manip_check = pd.read_csv(MANIP_CHECK_FILE)
_pool_check = _manip_check[_manip_check["sample"] == "pool"]
print(f"manipulation check of notebook 06: {len(_pool_check)} rows of the pool")
_words = _pool_check[["occ", "bio_idx", "arm", "delta_words", "n_replaced",
                      "n_deleted"]]
_model_frame = df_arm3_delta.merge(_words, on=["occ", "bio_idx", "arm"],
                                   how="left")
_no_words = int(_model_frame["delta_words"].isna().sum())
print(f"control biographies without a change in length: {_no_words} of "
      f"{len(_model_frame)} (expected 0)")
if _no_words:
    raise AssertionError(
        f"{_no_words} biographies of Arm 3 do not appear in the manipulation "
        f"check of notebook 06, so the change in length is unknown. The two "
        f"runs do not cover the same pool.")

_word_rows = []
print()
print(f"  {'arm':<10} {'n':>5} {'b_delta_words':>14} {'SE':>8} {'p':>10} "
      f"{'R2':>7}")
print("  " + "-" * 60)
for arm in ("communion", "agency"):
    _s = _model_frame[_model_frame["arm"] == arm]
    _n_occ = int(_s["occ"].nunique())
    if len(_s) < 30 or _n_occ < 2 or _s["delta_words"].nunique() < 2:
        raise AssertionError(
            f"The model for {arm} cannot be fitted: {len(_s)} biographies, "
            f"{_n_occ} occupations, {_s['delta_words'].nunique()} different "
            f"changes in length.")
    _fit = smf.ols("delta ~ delta_words + C(occ)", data=_s).fit(cov_type="HC1")
    print(f"  {arm:<10} {int(_fit.nobs):>5} "
          f"{float(_fit.params['delta_words']):>+14.4f} "
          f"{float(_fit.bse['delta_words']):>8.4f} "
          f"{float(_fit.pvalues['delta_words']):>10.4g} "
          f"{float(_fit.rsquared):>7.3f}")
    _word_rows.append({"arm": arm, "y": "delta_logit",
                       "predictors": "delta_words+C(occ)", "n": int(_fit.nobs),
                       "b_delta_words": float(_fit.params["delta_words"]),
                       "se": float(_fit.bse["delta_words"]),
                       "p": float(_fit.pvalues["delta_words"]),
                       "r2": float(_fit.rsquared)})
df_arm3_words = pd.DataFrame(_word_rows)
df_arm3_words["interpretable"] = ARM3_INTERPRETABLE
print()
print("The length of a biography changes because the manipulation replaces or "
      "deletes words. A coefficient away from zero says that the judgement "
      "follows the length as well as the wording.")

manipulation check of notebook 06: 6500 rows of the pool
control biographies without a change in length: 0 of 1040 (expected 0)

  arm            n  b_delta_words       SE          p      R2
  ------------------------------------------------------------
  communion    520        -0.0048   0.0569     0.9327   0.053
  agency       520        +0.5693   0.1730  0.0009977   0.211

The length of a biography changes because the manipulation replaces or deletes words. A coefficient away from zero says that the judgement follows the length as well as the wording.


## 6. Conditional on the wording of the advertisement

- **Question:** Does the change depend on how communal or agentic the job
  description is?
- **Expectation:** the change is larger against job descriptions that match the
  manipulated dimension.
- **Test:** job description; Spearman within occupation with a pooled
  permutation test, the same procedure as T1 in notebook 06.
- **Status:** exploratory, fixed 2026-10-12. The sample of this arm holds 20 job
  descriptions per occupation, so a test that conditions on a property of the
  job description rests on 20 points per occupation. The corresponding test of
  notebook 06 rests on 250 and carries the pre-specified result.
- **Output:** `study2_arm3_conditional.csv`

In [15]:
# The mean change per job description against the density of that job description.
_jd = pd.read_csv(JD_LEVEL)
_jd["jd_id"] = [f"{o}_jd{int(j):03d}" for o, j in zip(_jd["occ"], _jd["jd_idx"])]
_available = [c for c in ("comm_dens", "ag_dens", "balance",
                          "comm_dens_cw", "ag_dens_cw", "balance_cw")
              if c in _jd.columns]
print(f"density columns available: {_available}")

_per_jd = df_arm3_pairs.merge(
    df_arm3_delta[["occ", "bio_idx", "arm", "sim_cp"]],
    on=["occ", "bio_idx", "arm"], how="left")
_rows = []
for _subset, _frame in (("gate", _per_jd[_per_jd["sim_cp"] >= PRIMARY_GATE]),
                        ("all", _per_jd)):
    _mean_by_jd = _frame.groupby(["occ", "arm", "jd_id"])["delta"].mean().reset_index()
    _joined = _mean_by_jd.merge(_jd[["jd_id"] + _available], on="jd_id", how="inner")
    for arm in ("communion", "agency"):
        _a = _joined[_joined["arm"] == arm]
        for _column in _available:
            _delta_by_occ, _dens_by_occ = {}, {}
            for occ, group in _a.groupby("occ"):
                group = group.dropna(subset=["delta", _column])
                if len(group) < 5:
                    continue
                _delta_by_occ[occ] = group["delta"].to_numpy()
                _dens_by_occ[occ] = group[_column].to_numpy()
            if len(_delta_by_occ) < 2:
                continue
            _tab, _pool = study2_core.jd_language_test(
                _delta_by_occ, _dens_by_occ, n_perm=N_PERM, seed=SEED)
            # The status follows the run: in the main run these tests are
            # secondary and fixed in advance, in the pilot they describe the
            # sample and carry no inference. The row says which density it
            # reads.
            _status = (f"{DELTA_STATUS}, without the D1 function words"
                       if _column.endswith("_cw") else DELTA_STATUS)
            print(f"  {_subset:<5} {arm:<10} change against {_column:<14} "
                  f"rho {_pool.get('rho', float('nan')):>+6.3f}  "
                  f"p {_pool.get('p_rho', float('nan')):>7.4f}  "
                  f"occupations {len(_delta_by_occ):>2}  {_status}")
            _rows.append({"subset": _subset, "arm": arm, "x": _column,
                          "status": _status, "n_occupations": len(_delta_by_occ),
                          "rho": float(_pool.get("rho", float("nan"))),
                          "p_rho": float(_pool.get("p_rho", float("nan"))),
                          "diff_hi_lo": float(_pool.get("diff_hi_lo", float("nan"))),
                          "p_diff": float(_pool.get("p_diff", float("nan")))})
df_arm3_conditional = pd.DataFrame(_rows)
df_arm3_conditional["interpretable"] = ARM3_INTERPRETABLE

density columns available: ['comm_dens', 'ag_dens', 'balance', 'comm_dens_cw', 'ag_dens_cw', 'balance_cw']
  gate  communion  change against comm_dens      rho -0.047  p  0.4497  occupations 13  secondary, fixed in advance
  gate  communion  change against ag_dens        rho -0.011  p  0.8632  occupations 13  secondary, fixed in advance
  gate  communion  change against balance        rho -0.074  p  0.2408  occupations 13  secondary, fixed in advance
  gate  communion  change against comm_dens_cw   rho -0.052  p  0.4065  occupations 13  secondary, fixed in advance, without the D1 function words
  gate  communion  change against ag_dens_cw     rho +0.005  p  0.9322  occupations 13  secondary, fixed in advance, without the D1 function words
  gate  communion  change against balance_cw     rho -0.065  p  0.2981  occupations 13  secondary, fixed in advance, without the D1 function words
  gate  agency     change against comm_dens      rho -0.030  p  0.6387  occupations 13  secondary, fixed

## 7. The two tests fixed in advance

- **Question:** T1 - does the gap between women and men follow the wording of
  the advertisement? T2 - does a biography that the model rates higher against
  communal advertisements also carry the more communal wording?
- **Expectation:** both correlations positive, the sign fixed in advance.
- **Test:** T1 on the job description: the gap women minus men over the original
  biographies, against the balance of the advertisement under dictionary A. T2
  on the biography: the slope of its value over the advertisements of its
  occupation against their balance, then against the balance of the biography.
  Spearman within the occupation with a pooled permutation test in both cases,
  the procedure of T1 and T2 in notebook 06, on both measures of the value.
- **Status:** in the main run the two tests fixed in advance, each at
  alpha = 0.025. The pilot holds 8 advertisements per occupation and carries no
  inference of its own; there the numbers describe the sample.
- **Output:** `study2_arm3_t1.csv`, `study2_arm3_t2.csv`,
  `study2_arm3_gender_gap.csv`, `study2_arm3_slopes.csv`

In [16]:
# T1 of this arm: does the gap between women and men follow the wording of the
# advertisement? The gap is the mean value of the original biographies of women
# minus that of men, per job description, and it is tested against the balance
# of that advertisement under dictionary A. Spearman within the occupation with
# a pooled permutation test, the procedure of T1 in notebook 06, on both
# measures: the value itself and the percentile rank.
#
# In the main run this is one of the two tests fixed in advance, each at
# alpha = 0.025. The pilot holds 8 advertisements per occupation and carries no
# inference of its own; its numbers are printed as a description.
_jd_absent = sorted(set(df_arm3_originals["jd_id"]) - set(_jd["jd_id"]))
print(f"control advertisements of this run that notebook 06 does not hold: "
      f"{len(_jd_absent)} (expected 0)")
if _jd_absent:
    raise AssertionError(
        f"{len(_jd_absent)} advertisements of this run are not in "
        f"{JD_LEVEL.name}, for example {_jd_absent[:5]}; without their balance "
        f"the two tests would silently rest on fewer advertisements.")
_t1_rows = []
_gap_frames = {}
for _measure, _column in (("logit", "logit"), ("percentile", "pct_orig")):
    _gap_rows, _gap_left_out = [], []
    for (occ, jd_id), group in df_arm3_originals.groupby(["occ", "jd_id"]):
        _f = group[group["female"] == 1][_column]
        _m = group[group["female"] == 0][_column]
        if len(_f) < 3 or len(_m) < 3:
            _gap_left_out.append((occ, jd_id, len(_f), len(_m)))
            continue
        _gap_rows.append({"occ": occ, "jd_id": jd_id, "measure": _measure,
                          "gap_f_minus_m": float(_f.mean() - _m.mean()),
                          "n_f": len(_f), "n_m": len(_m)})
    _gaps = pd.DataFrame(_gap_rows).merge(_jd[["jd_id", "balance"]],
                                          on="jd_id", how="inner")
    if _gaps.empty:
        raise AssertionError(
            f"No job description has at least three biographies of each gender "
            f"with a value, so the gap in {_measure} cannot be formed.")
    _gap_frames[_measure] = _gaps
    _gap_by_occ = {o: g["gap_f_minus_m"].to_numpy()
                   for o, g in _gaps.groupby("occ") if len(g) >= 5}
    _bal_by_occ = {o: g["balance"].to_numpy()
                   for o, g in _gaps.groupby("occ") if len(g) >= 5}
    if _gap_left_out:
        print(f"  advertisements with fewer than three biographies of a gender, "
              f"left out: {len(_gap_left_out)} {_gap_left_out[:5]}")
    print(f"{_measure}: job descriptions with a gap {len(_gaps)}, "
          f"occupations in the test {len(_gap_by_occ)}, "
          f"mean gap {_gaps['gap_f_minus_m'].mean():+.3f}")
    if len(_gap_by_occ) < 2:
        raise AssertionError(
            f"Only {len(_gap_by_occ)} occupations have at least 5 job "
            f"descriptions with a gap; the pooled test needs 2.")
    _tab, _pool = study2_core.jd_language_test(
        _gap_by_occ, _bal_by_occ, n_perm=N_PERM, seed=SEED)
    _rho_pooled, _p_pooled = spearmanr(_gaps["gap_f_minus_m"], _gaps["balance"])
    _row = {"test": "T1_gap_balance", "measure": _measure,
            "unit": "job description", "y": "gap_f_minus_m", "x": "balance",
            "status": T1_T2_STATUS, "alpha": ALPHA_PRE,
            "n_occupations": len(_gap_by_occ), "n_jds": len(_gaps),
            "expected_sign": "+",
            "rho_within_occ": float(_pool.get("rho", float("nan"))),
            "p_within_occ": float(_pool.get("p_rho", float("nan"))),
            "diff_hi_lo": float(_pool.get("diff_hi_lo", float("nan"))),
            "p_diff": float(_pool.get("p_diff", float("nan"))),
            "rho_over_all_jds": float(_rho_pooled),
            "p_over_all_jds": float(_p_pooled)}
    print(f"  within occupation: rho {_row['rho_within_occ']:+.3f}  "
          f"p {_row['p_within_occ']:.4f}  "
          f"(alpha {ALPHA_PRE}, {T1_T2_STATUS})")
    print(f"  over all job descriptions: rho {float(_rho_pooled):+.3f}  "
          f"p {float(_p_pooled):.4f}")
    if INFERENCE:
        print(f"  decision at alpha {ALPHA_PRE}: "
              f"{'below' if _row['p_within_occ'] < ALPHA_PRE else 'not below'} "
              f"the threshold")
    else:
        print(f"  the pilot carries no inference; the numbers describe the "
              f"sample of {len(_gaps)} advertisements")
    _t1_rows.append(_row)
df_arm3_gender_gap = pd.concat(_gap_frames.values(), ignore_index=True).merge(
    pd.DataFrame(_t1_rows)[["measure", "rho_within_occ", "p_within_occ",
                            "status"]], on="measure", how="left")
df_arm3_t1 = pd.DataFrame(_t1_rows)
df_arm3_t1["interpretable"] = ARM3_INTERPRETABLE

control advertisements of this run that notebook 06 does not hold: 0 (expected 0)
logit: job descriptions with a gap 260, occupations in the test 13, mean gap +1.408
  within occupation: rho +0.106  p 0.0970  (alpha 0.025, pre-specified)
  over all job descriptions: rho +0.250  p 0.0000
  decision at alpha 0.025: not below the threshold
percentile: job descriptions with a gap 260, occupations in the test 13, mean gap +3.441
  within occupation: rho +0.132  p 0.0330  (alpha 0.025, pre-specified)
  over all job descriptions: rho +0.246  p 0.0001
  decision at alpha 0.025: not below the threshold


In [17]:
# T2 of this arm: does a biography that the model rates higher against communal
# advertisements also carry the more communal wording? Per biography the slope
# of its value over the advertisements of its occupation against their balance,
# then that slope against the balance of the biography itself. Spearman within
# the occupation with a pooled permutation test, the procedure of T2 in
# notebook 06, on both measures.
#
# In the main run this is the second of the two tests fixed in advance, at
# alpha = 0.025.
_bio_balance = pd.read_csv(BIO_SLOPES_FILE)[["occ", "bio_idx", "bio_balance"]]
print(f"biography balance from notebook 06: {len(_bio_balance)} biographies")
_balance_of_jd = _jd.set_index("jd_id")["balance"]

_t2_rows, _slope_rows = [], []
for _measure, _column in (("logit", "logit"), ("percentile", "pct_orig")):
    _rows, _slope_left_out = [], []
    for (occ, bio_idx), group in df_arm3_originals.groupby(["occ", "bio_idx"]):
        _x = _balance_of_jd.reindex(group["jd_id"]).to_numpy(float)
        _y = group[_column].to_numpy(float)
        if np.isnan(_x).any():
            raise AssertionError(
                f"{occ}/{bio_idx}: an advertisement has no balance, although "
                f"the control of T1 found none missing.")
        if len(_y) < 5:
            _slope_left_out.append((occ, int(bio_idx), len(_y)))
            continue
        _centred = _x - _x.mean()
        _denominator = float(_centred @ _centred)
        if _denominator < 1e-12:
            raise AssertionError(
                f"The balance of the advertisements of {occ} does not vary, so "
                f"no slope can be formed on it.")
        _rows.append({"occ": occ, "bio_idx": int(bio_idx), "measure": _measure,
                      "n_jds": len(_y), "female": int(group["female"].iloc[0]),
                      "slope": float((_y - _y.mean()) @ _centred / _denominator)})
    if _slope_left_out:
        print(f"  biographies rated against fewer than five advertisements, "
              f"left out: {len(_slope_left_out)} {_slope_left_out[:5]}")
    _slopes = pd.DataFrame(_rows).merge(_bio_balance, on=["occ", "bio_idx"],
                                        how="left")
    _without = int(_slopes["bio_balance"].isna().sum())
    print(f"{_measure}: biographies with a slope {len(_slopes)}, "
          f"without a balance from notebook 06: {_without} (expected 0)")
    if _without:
        raise AssertionError(
            f"{_without} biographies of Arm 3 do not appear in "
            f"{BIO_SLOPES_FILE.name}, so their balance is unknown; the two runs "
            f"do not cover the same pool.")
    _slope_rows.append(_slopes)
    _slope_by_occ = {o: g["slope"].to_numpy()
                     for o, g in _slopes.groupby("occ") if len(g) >= 5}
    _bal_by_occ = {o: g["bio_balance"].to_numpy()
                   for o, g in _slopes.groupby("occ") if len(g) >= 5}
    if len(_slope_by_occ) < 2:
        raise AssertionError(
            f"Only {len(_slope_by_occ)} occupations have at least 5 biographies "
            f"with a slope; the pooled test needs 2.")
    _tab, _pool = study2_core.jd_language_test(
        _slope_by_occ, _bal_by_occ, n_perm=N_PERM, seed=SEED)
    _rho_pooled, _p_pooled = spearmanr(_slopes["slope"], _slopes["bio_balance"])
    _row = {"test": "T2_slope_balance", "measure": _measure,
            "unit": "biography", "y": "slope_over_jds", "x": "bio_balance",
            "status": T1_T2_STATUS, "alpha": ALPHA_PRE,
            "n_occupations": len(_slope_by_occ), "n_bios": len(_slopes),
            "expected_sign": "+",
            "rho_within_occ": float(_pool.get("rho", float("nan"))),
            "p_within_occ": float(_pool.get("p_rho", float("nan"))),
            "diff_hi_lo": float(_pool.get("diff_hi_lo", float("nan"))),
            "p_diff": float(_pool.get("p_diff", float("nan"))),
            "rho_over_all_bios": float(_rho_pooled),
            "p_over_all_bios": float(_p_pooled)}
    print(f"  within occupation: rho {_row['rho_within_occ']:+.3f}  "
          f"p {_row['p_within_occ']:.4f}  "
          f"(alpha {ALPHA_PRE}, {T1_T2_STATUS})")
    print(f"  over all biographies: rho {float(_rho_pooled):+.3f}  "
          f"p {float(_p_pooled):.4f}")
    if INFERENCE:
        print(f"  decision at alpha {ALPHA_PRE}: "
              f"{'below' if _row['p_within_occ'] < ALPHA_PRE else 'not below'} "
              f"the threshold")
    else:
        print(f"  the pilot carries no inference; the numbers describe the "
              f"sample of {len(_slopes)} biographies")
    _t2_rows.append(_row)
df_arm3_slopes = pd.concat(_slope_rows, ignore_index=True)
df_arm3_t2 = pd.DataFrame(_t2_rows)
df_arm3_t2["interpretable"] = ARM3_INTERPRETABLE

biography balance from notebook 06: 3250 biographies
logit: biographies with a slope 1283, without a balance from notebook 06: 0 (expected 0)
  within occupation: rho +0.076  p 0.0052  (alpha 0.025, pre-specified)
  over all biographies: rho +0.064  p 0.0229
  decision at alpha 0.025: below the threshold
percentile: biographies with a slope 1283, without a balance from notebook 06: 0 (expected 0)
  within occupation: rho +0.085  p 0.0022  (alpha 0.025, pre-specified)
  over all biographies: rho +0.086  p 0.0021
  decision at alpha 0.025: below the threshold


## 8. Comparison of the arms

- **Question:** Do the three arms move the same biographies in the same
  direction, and how well does each of them tell the occupation apart?
- **Test:** job description; the area under the curve of Arm 1 and Arm 2 on the
  same pairs as Arm 3, read from the embeddings of notebook 06 and of Kaggle.
  Biography; Spearman between the change in this arm and the change in Arm 1 and
  Arm 2 on the same biographies.
- **Status:** robustness, fixed 2026-10-11; the comparison of the curves added
  2026-10-12.
- **Output:** `study2_arm_auc_comparison.csv`,
  `study2_arm3_arm_comparison.csv`

In [18]:
# The same curve for Arm 1 and Arm 2, on the same pairs: how well the similarity
# of the two embedding arms separates the biographies of the occupation from
# those of other occupations, for every job description of the Arm 3 sample.
# Arm 1 reads the embeddings of notebook 06 from its cache, Arm 2 the
# embeddings produced on Kaggle. Nothing is embedded here.
_arm1_cache = Path(CACHE_BASE) / PRIMARY_MODEL.replace("/", "_").replace("-", "_")
_jd_pairs = raw[(raw["version"] == "original") & raw["has_value"]][
    ["jd_id", "occ", "bio_occ", "bio_idx", "cross_occupation"]].drop_duplicates()
_occupations = sorted(_jd_pairs["occ"].unique())
_bio_occupations = sorted(_jd_pairs["bio_occ"].unique())

_absent = [f"{occ}_pool_orig" for occ in _bio_occupations
           if not (_arm1_cache / f"{occ}_pool_orig.npy").exists()]
_absent += [f"{occ}_jds" for occ in _occupations
            if not (_arm1_cache / f"{occ}_jds.npy").exists()]
print(f"Arm 1 cache: {_arm1_cache}")
print(f"control arrays of Arm 1 that are missing: {len(_absent)} (expected 0)")
if _absent:
    raise AssertionError(
        f"The cache of notebook 06 does not hold {_absent}. The comparison reads "
        f"the embeddings of the full run; run notebook 06 with SMOKE = False "
        f"first.")

_bio_emb_1 = {occ: np.load(_arm1_cache / f"{occ}_pool_orig.npy",
                           allow_pickle=True).item()["mean"]
              for occ in _bio_occupations}
_jd_emb_1 = {occ: np.load(_arm1_cache / f"{occ}_jds.npy", allow_pickle=True).item()
             for occ in _occupations}

_arm2_bio = pd.read_parquet(ARM2_BIO_FILE)
_arm2_jd = pd.read_parquet(ARM2_JD_FILE)
_bio_emb_2 = {(r.occ, int(r.bio_idx)): np.asarray(r.embedding)
              for r in _arm2_bio[_arm2_bio["version"] == "original"].itertuples()}
_jd_emb_2 = {(r.occ, int(r.jd_idx)): np.asarray(r.embedding)
             for r in _arm2_jd.itertuples()}

def _jd_index(jd_id):
    return int(str(jd_id).rsplit("jd", 1)[1])

_out_of_range = [jd for jd in sorted(_jd_pairs["jd_id"].unique())
                 if _jd_index(jd) >= len(_jd_emb_1[
                     _jd_pairs[_jd_pairs["jd_id"] == jd]["occ"].iloc[0]]["mean"])]
print(f"control job descriptions outside the pool of their occupation: "
      f"{len(_out_of_range)} (expected 0)")
if _out_of_range:
    raise AssertionError(
        f"{_out_of_range} lie outside the job description pool of notebook 06, "
        f"so the embeddings do not belong to the same sample.")

_auc_arm_rows = []
for jd_id, group in _jd_pairs.groupby("jd_id"):
    _occ = group["occ"].iloc[0]
    _jd_idx = _jd_index(jd_id)
    _own = group[~group["cross_occupation"]]
    _other = group[group["cross_occupation"]]
    if len(_own) < 5 or len(_other) < 5:
        print(f"  {jd_id}: {len(_own)} own and {len(_other)} other biographies, "
              f"5 of each needed; left out")
        continue
    _labels = np.r_[np.ones(len(_own)), np.zeros(len(_other))]
    for _arm_label in ("arm1", "arm2"):
        if _arm_label == "arm1":
            _jd_vector = _jd_emb_1[_occ]["mean"][_jd_idx]
            _scores = np.array([
                float(_bio_emb_1[r.bio_occ][int(r.bio_idx)] @ _jd_vector)
                for r in pd.concat([_own, _other]).itertuples()])
        else:
            _jd_vector = _jd_emb_2[(_occ, _jd_idx)]
            _scores = np.array([
                float(_bio_emb_2[(r.bio_occ, int(r.bio_idx))] @ _jd_vector)
                for r in pd.concat([_own, _other]).itertuples()])
        if len(np.unique(_scores)) < 2:
            raise AssertionError(
                f"{_arm_label} gives the same similarity to every biography of "
                f"{jd_id}, so the curve is not defined.")
        _auc_arm_rows.append({"arm_label": _arm_label, "jd_id": jd_id,
                              "occ": _occ, "n_own": len(_own),
                              "n_other": len(_other),
                              "auc": float(roc_auc_score(_labels, _scores))})
df_arm_auc = pd.DataFrame(_auc_arm_rows)
if df_arm_auc.empty:
    raise AssertionError("No job description entered the comparison of the arms.")

# Arm 3 on the same job descriptions, so the three numbers are comparable.
_arm3_same = df_arm3_auc[df_arm3_auc["jd_id"].isin(df_arm_auc["jd_id"])]
df_arm_auc = pd.concat([df_arm_auc,
                        _arm3_same.assign(arm_label="arm3")[df_arm_auc.columns]],
                       ignore_index=True)
print()
print(f"  {'arm':<6} {'jds':>5} {'mean':>7} {'median':>7} {'min':>7} {'max':>7} "
      f"{'share below 0.5':>16}")
print("  " + "-" * 60)
for _arm_label in ("arm1", "arm2", "arm3"):
    _s = df_arm_auc[df_arm_auc["arm_label"] == _arm_label]["auc"]
    print(f"  {_arm_label:<6} {len(_s):>5} {_s.mean():>7.3f} {_s.median():>7.3f} "
          f"{_s.min():>7.3f} {_s.max():>7.3f} {(_s < 0.5).mean():>15.1%}")
print()
print(f"The minimum fixed in advance, {AUC_MINIMUM}, applies to Arm 3. The two "
      f"embedding arms are reported next to it as a measure of comparison.")

Arm 1 cache: outputs\study2_cache\sentence_transformers_all_mpnet_base_v2
control arrays of Arm 1 that are missing: 0 (expected 0)
control job descriptions outside the pool of their occupation: 0 (expected 0)

  arm      jds    mean  median     min     max  share below 0.5
  ------------------------------------------------------------
  arm1     260   0.776   0.825   0.199   0.985            7.3%
  arm2     260   0.779   0.804   0.284   0.973            5.0%
  arm3     260   0.838   0.883   0.356   0.994            4.6%

The minimum fixed in advance, 0.75, applies to Arm 3. The two embedding arms are reported next to it as a measure of comparison.


In [19]:
# The change here against the change of the two embedding arms.
_frame = df_arm3_delta.merge(
    pd.read_csv(ARM2_FILE)[["occ", "bio_idx", "arm", "delta"]]
    .rename(columns={"delta": "delta_arm2"}),
    on=["occ", "bio_idx", "arm"], how="left")
print(f"control biographies with an Arm 2 change: "
      f"{int(_frame['delta_arm2'].notna().sum())} of {len(_frame)}")

print()
print(f"{'arm':<10} {'against':<8} {'subset':<7} {'n':>5} {'rho':>7} {'p':>10}")
print("-" * 52)
_comparison_rows = []
for arm in ("communion", "agency"):
    for other, column in (("arm1", "delta_arm1"), ("arm2", "delta_arm2")):
        for subset, sub in (("gate", _frame[_frame["sim_cp"] >= PRIMARY_GATE]),
                            ("all", _frame)):
            _s = sub[sub["arm"] == arm].dropna(subset=["delta", column])
            if len(_s) < 10:
                continue
            _rho, _p = spearmanr(_s["delta"], _s[column])
            print(f"{arm:<10} {other:<8} {subset:<7} {len(_s):>5} {_rho:>+7.3f} "
                  f"{_p:>10.4g}")
            _comparison_rows.append({"arm": arm, "against": other, "subset": subset,
                                     "n": len(_s), "rho": float(_rho),
                                     "p": float(_p)})
df_arm3_arm_comparison = pd.DataFrame(_comparison_rows)
df_arm3_arm_comparison["interpretable"] = ARM3_INTERPRETABLE

control biographies with an Arm 2 change: 1040 of 1040

arm        against  subset      n     rho          p
----------------------------------------------------
communion  arm1     gate      519  -0.045     0.3019
communion  arm1     all       520  -0.043     0.3318
communion  arm2     gate      519  +0.033     0.4505
communion  arm2     all       520  +0.036     0.4132
agency     arm1     gate      517  +0.147   0.000774
agency     arm1     all       520  +0.150  0.0005966
agency     arm2     gate      517  +0.123   0.004991
agency     arm2     all       520  +0.122   0.005345


## 9. Save outputs

In [20]:
# Write every result table. The tables are listed by name: a name that no part
# of the notebook has filled stops the run, an empty table stops it too, and at
# the end a control lists every file that was written with its number of rows.
ARM3_OUT.mkdir(parents=True, exist_ok=True)
TABLES = [
    ("study2_arm3_validity.csv",        "df_arm3_validity"),
    ("study2_arm3_sessions.csv",        "df_arm3_sessions"),
    ("study2_arm3_originals.csv",       "df_arm3_originals"),
    ("study2_arm3_t1.csv",              "df_arm3_t1"),
    ("study2_arm3_t2.csv",              "df_arm3_t2"),
    ("study2_arm3_slopes.csv",          "df_arm3_slopes"),
    ("study2_arm3_auc_by_jd.csv",       "df_arm3_auc"),
    ("study2_arm_auc_comparison.csv",   "df_arm_auc"),
    ("study2_arm3_delta.csv",           "df_arm3_delta"),
    ("study2_arm3_delta_by_jd.csv",     "df_arm3_pairs"),
    ("study2_arm3_pct.csv",             "df_arm3_pct"),
    ("study2_arm3_pct_by_jd.csv",       "df_arm3_pct_by_jd"),
    ("study2_arm3_tests.csv",           "df_arm3_tests"),
    ("study2_arm3_switch.csv",          "df_arm3_switch"),
    ("study2_arm3_split.csv",           "df_arm3_split"),
    ("study2_arm3_words.csv",           "df_arm3_words"),
    ("study2_arm3_conditional.csv",     "df_arm3_conditional"),
    ("study2_arm3_gender_gap.csv",      "df_arm3_gender_gap"),
    ("study2_arm3_arm_comparison.csv",  "df_arm3_arm_comparison"),
]
_never_built = [name for _, name in TABLES if name not in globals()]
if _never_built:
    raise AssertionError(
        f"These tables were never built: {_never_built}. The parts that fill "
        f"them have to run before this cell.")
_no_rows = [name for _, name in TABLES if globals()[name].empty]
if _no_rows:
    raise AssertionError(
        f"These tables hold no rows: {_no_rows}. An empty table is not a "
        f"result, so the run stops here rather than writing an empty file.")

written = {}
for _file, _name in TABLES:
    globals()[_name].to_csv(ARM3_OUT / _file, index=False, encoding="utf-8")
    written[_file] = len(globals()[_name])

print(f"  {'file':<42} {'rows':>7}  {'KB':>8}")
print("  " + "-" * 62)
_failed = []
for _file in sorted(written):
    _path = ARM3_OUT / _file
    if not _path.exists() or _path.stat().st_size == 0 or written[_file] == 0:
        _failed.append(_file)
        continue
    print(f"  {_file:<42} {written[_file]:>7} {_path.stat().st_size / 1024:>8.1f}")
if _failed:
    raise AssertionError(f"Missing or empty after writing: {_failed}.")
print()
print(f"files written: {len(written)} tables in {ARM3_OUT}/")
print(f"run: {ARM3_RUN}  inference: {INFERENCE}  "
      f"status of the two tests fixed in advance: {T1_T2_STATUS}")
print(f"mean area under the curve of Arm 3: {_mean_auc:.3f}; arm interpretable: "
      f"{ARM3_INTERPRETABLE}")

  file                                          rows        KB
  --------------------------------------------------------------
  study2_arm3_arm_comparison.csv                   8      0.6
  study2_arm3_auc_by_jd.csv                      260     11.4
  study2_arm3_conditional.csv                     24      3.8
  study2_arm3_delta.csv                         1040    162.5
  study2_arm3_delta_by_jd.csv                  20772   2022.6
  study2_arm3_gender_gap.csv                     520     67.6
  study2_arm3_originals.csv                    25660   1752.1
  study2_arm3_pct.csv                           1027     87.3
  study2_arm3_pct_by_jd.csv                    20540   1810.8
  study2_arm3_sessions.csv                         1      0.2
  study2_arm3_slopes.csv                        2566    157.0
  study2_arm3_split.csv                            6      0.7
  study2_arm3_switch.csv                           2      0.2
  study2_arm3_t1.csv                               2      0.6
  st